# 供应链运营深度分析报告
## DataCo Global Supply Chain — 180K 订单 | SQL + Python + Streamlit

本 Notebook 作为 Streamlit 仪表板的补充，提供更深层的统计分析和业务洞察。

In [ ]:
import sqlite3, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size':12, 'figure.dpi':120, 'figure.figsize':(12,5)})
sns.set_style('whitegrid')

conn = sqlite3.connect('../data/supply_chain.db')
df = pd.read_sql("""
    SELECT f.*, s.shipping_mode, c.market, c.order_region,
           p.category_name, p.department_name
    FROM fact_orders f
    JOIN dim_shipping s ON f.shipping_mode_id=s.shipping_mode_id
    JOIN dim_customer c ON f.customer_id=c.customer_id
    JOIN dim_product p ON f.product_card_id=p.product_card_id
    WHERE f.order_status='COMPLETE'
""", conn)

df['order_date'] = pd.to_datetime(df['order_date'])
print(f'{len(df):,} 行 | {df.order_id.nunique():,} 单 | {df.order_date.min().date()} ~ {df.order_date.max().date()}')

## 1. 全局准时率概览

In [ ]:
on_time = df['on_time'].mean() * 100
delayed = df['is_delayed'].sum()
avg_delay = df.loc[df['is_delayed']==1, 'delay_days'].mean()
risk = df.loc[df['is_delayed']==1, 'order_total_value'].sum()

print(f'准时率: {on_time:.1f}%')
print(f'延迟订单行: {delayed:,} ({delayed/len(df)*100:.1f}%)')
print(f'平均延迟: {avg_delay:.1f} 天')
print(f'延迟收入风险: ${risk:,.0f}')

## 2. 运输方式深度对比
First Class 准时率 0% 是数据中最显著的异常信号。分析 SLA 合理性。

In [ ]:
mode_stats = df.groupby('shipping_mode').agg(
    orders=('order_id','nunique'), ot=('on_time','mean'),
    avg_real=('days_shipping_real','mean'), avg_sched=('days_shipping_scheduled','mean'),
    delay_rate=('is_delayed','mean'), revenue=('order_total_value','sum')
).reset_index()
mode_stats['sla_gap'] = mode_stats['avg_real'] - mode_stats['avg_sched']
mode_stats['ot'] *= 100
mode_stats['delay_rate'] *= 100
mode_stats[['shipping_mode','orders','ot','avg_sched','avg_real','sla_gap','delay_rate']].round(2)

**关键发现**: First Class 承诺1天交付，实际平均2.0天，形成系统性SLA违约。SLA设置不合理，建议调整为2天承诺或优化运营使其达到1天。

## 3. 市场区域绩效诊断

In [ ]:
mkt = df.groupby('market').agg(ot=('on_time','mean'), orders=('order_id','nunique'),
    avg_delay=('delay_days','mean'), revenue=('order_total_value','sum')).reset_index()
mkt['ot'] *= 100
global_avg = mkt['ot'].mean()
mkt['vs_global'] = mkt['ot'] - global_avg
mkt.sort_values('ot', ascending=False).round(2)

## 4. 延迟根因归因分析
通过极差分析识别最强延迟驱动因素

In [ ]:
factors = {}
for col in ['shipping_mode','market','order_weekday','order_quarter','department_name','category_name']:
    ot = df.groupby(col)['on_time'].mean() * 100
    factors[col] = {'range': ot.max() - ot.min(), 'best': ot.idxmax(), 'worst': ot.idxmin(),
                    'best_val': ot.max(), 'worst_val': ot.min()}

factor_df = pd.DataFrame(factors).T.sort_values('range', ascending=False)
factor_df.round(2)

## 5. 延迟集中度分析
多少比例的延迟可以通过解决轻微延迟来消除？

In [ ]:
delayed = df[df['is_delayed']==1]
bins = [0, 1, 3, 7, 14, np.inf]
labels = ['1天','2-3天','4-7天','8-14天','15天+']
delayed['delay_bin'] = pd.cut(delayed['delay_days'], bins=bins, labels=labels, right=True)
dist = delayed['delay_bin'].value_counts().sort_index()
dist_pct = (dist / dist.sum() * 100).round(1)
cum_pct = dist_pct.cumsum()

for i, (cat, cnt, pct, cum) in enumerate(zip(dist.index, dist.values, dist_pct.values, cum_pct.values)):
    bar = '█' * int(pct/2)
    print(f'{cat:10s} {cnt:>8,}条  {pct:>5.1f}%  累计{cum:>5.1f}%  {bar}')

## 6. 月度趋势与季节性检验

In [ ]:
monthly = df.groupby('year_month').agg(ot=('on_time','mean'), orders=('order_id','nunique')).reset_index()
monthly['ot'] *= 100
monthly['rolling_3'] = monthly['ot'].rolling(3, min_periods=1).mean()

fig, axes = plt.subplots(2,1,figsize=(14,8))

# 月度趋势
axes[0].plot(monthly.index, monthly['ot'], alpha=0.5, label='月准时率')
axes[0].plot(monthly.index, monthly['rolling_3'], linewidth=2, label='3月滚动均值', color='#ff7f0e')
axes[0].axhline(y=50, ls='--', color='gray', alpha=0.5)
axes[0].legend(); axes[0].set_ylabel('准时率 (%)'); axes[0].set_title('月度准时率趋势')

# 月度季节性（均值）
df['order_month'] = pd.to_datetime(df['order_date']).dt.month
seasonal = df.groupby('order_month').agg(ot=('on_time','mean')).reset_index()
seasonal['ot'] *= 100
axes[1].bar(seasonal['order_month'], seasonal['ot'], color='#1f77b4')
axes[1].set_xticks(range(1,13))
axes[1].axhline(y=seasonal['ot'].mean(), ls='--', color='gray', alpha=0.5)
axes[1].set_xlabel('月份'); axes[1].set_ylabel('准时率 (%)'); axes[1].set_title('月度季节性模式')
plt.tight_layout()
plt.show()

## 7. 优化建议与ROI估算

In [ ]:
total_revenue = df['order_total_value'].sum()
at_risk = df.loc[df['is_delayed']==1, 'order_total_value'].sum()
print(f'总收入: ${total_revenue:,.0f}')
print(f'延迟风险敞口: ${at_risk:,.0f} ({at_risk/total_revenue*100:.1f}%)')
print()

# 场景分析
improvements = [5, 10, 15]
current_ot = df['on_time'].mean() * 100
for imp in improvements:
    recovered = at_risk * imp / 100
    new_ot = current_ot + (100-current_ot) * imp / 100
    print(f'  准时率提升 {imp}pp → 可回收约 ${recovered:,.0f} → 新准时率约 {new_ot:.1f}%')

print()

# Second Class 改进
sc = df[df['shipping_mode']=='Second Class']
sc_delayed = sc['is_delayed'].sum()
sc_revenue_risk = sc.loc[sc['is_delayed']==1, 'order_total_value'].sum()
print(f'Second Class — {sc_delayed:,}条延迟, 风险${sc_revenue_risk:,.0f}')
print(f'若将Second Class SLA从2天调整为4天: 准时率从{sc["on_time"].mean()*100:.1f}% → {(sc["days_shipping_real"] <= 4).mean()*100:.1f}%')

# First Class 改进
fc = df[df['shipping_mode']=='First Class']
print(f'First Class — 若SLA调为2天: 准时率从0% → {(fc["days_shipping_real"] <= 2).mean()*100:.1f}%')

## 结论

1. **运输方式是延迟最强驱动因素**，影响强度远超市场区域和季节因素
2. **First Class SLA不合理**（承诺1天/实需2天），调整SLA即可消除100%的系统性违约
3. **Second Class 严重低效**（承诺2天/实需3.9天），需运营审计
4. **Standard Class 是唯一可靠选项**，但速度最慢（4天），适合非紧急订单
5. **超半数延迟为1-3天轻微延迟**，流程优化ROI最高
6. **LATAM市场基础设施瓶颈**明显，需区域策略调整